# 01 — Chargement FINESS+

Charge les personnes morales et les etablissements depuis
`BIFINESS_DWH_SNAPSHOT`, avec leur adresse.

Deux differences avec l'ancien modele. L'adresse n'est plus une colonne de la
table structure : elle est atteinte par une table de liaison, la requete
enchaine donc structure, liaison, adresse. Et les tables sont historisees : le
filtre `BL_VersionCourante = 1` est indispensable, sans lui chaque structure
remonte autant de fois qu'elle a de versions.

Sortie : deux parquets dans `data/interim/`.

In [1]:
%pip install -r ../requirements.txt --quiet


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
from src.connexion import get_finess_connection
from src.chargement import (
    resoudre_tables, charger_pm, charger_ege, dedupliquer,
    charger_dates_ouverture_pm, colonnes_disponibles,
)
from src.display import afficher_tableau
from config.settings import FINESS_PM_RAW, FINESS_EGE_RAW, EVENEMENTS_PM, INTERIM_DIR

INTERIM_DIR.mkdir(parents=True, exist_ok=True)

# >>> A AJUSTER : False pour charger aussi les structures fermees
ACTIFS_SEULEMENT = True

conn = get_finess_connection()

TABLES = resoudre_tables(conn)


   pm           -> T_FAC_PmSmsse
   ege          -> T_FAC_Ege
   adresse      -> T_FAC_Adresse
   pm_adresse   -> T_FAC_PmSmsseAdresse
   ege_adresse  -> T_FAC_EgeAdresse
   evenement    -> T_FAC_Evenement
   telecom      -> T_FAC_Telecom
   pm_contact   -> T_FAC_PmSmsseContact
   ege_contact  -> T_FAC_EgeContact


In [3]:
# En cas d'echec sur une colonne inconnue :


## 1. Personnes morales

Une PM ne devrait porter qu'une adresse, celle du siege. La deduplication ne
fait donc rien si c'est bien le cas ; si elle intervient, le message le signale.

In [4]:
df_pm = charger_pm(conn, TABLES, actifs_seulement=ACTIFS_SEULEMENT)
print(f"PM chargees : {len(df_pm):,}")

# >>> A AJUSTER : si TX_UsageAdresse distingue une adresse principale, indiquer
#     sa valeur pour privilegier ce type plutot que la premiere venue.
df_pm = dedupliquer(df_pm, "NB_PmSmsseId", usage_prioritaire=None)

print(f"   avec SIREN   : {df_pm['TX_Siren'].notna().sum():,}")
print(f"   avec adresse : {df_pm['TX_CogCommune'].notna().sum():,}")

df_pm.to_parquet(FINESS_PM_RAW, index=False)
print(f"Sauvegarde : {FINESS_PM_RAW}")
afficher_tableau(df_pm, "Apercu PM FINESS+")

PM chargees : 56,193
   avec SIREN   : 54,103
   avec adresse : 56,193
Sauvegarde : /home/jovyan/work/finess+_refonte/projet_finess_sirene_finess+/data/interim/finess_pm.parquet


ID_PmSmsse,NB_PmSmsseId,TX_NumFinessPm,TX_Siren,TX_DenominationPm,TX_DenominationLonguePmSmsse,TX_ComplementAdressePmSmsse,TX_CategorieEntiteGeographiqueExercice,TX_StatutJuridique,TX_TypePersonneMorale,TX_EtatActif,FK_MetadonneeId,NB_EntrepriseInseeId,TX_UsageAdresse,NB_AdresseId,TX_TypeAdresse,TX_NumeroVoie,TX_TypeVoie,TX_LibelleVoie,TX_ComplementPointGeographique,TX_LieuDit,TX_CodePostal,TX_Localite,TX_CogCommune
11112,2,010000156,772201489,CLINIQUE CONVERT,CLINIQUE CONVERT,None,None,73,01,A,20,16070511.000000,02,2,None,62,AV,DE JASSERON,None,None,01000,BOURG EN BRESSE,01053
55556,6,010000222,546120122,CLINIQUE DU SOUFFLE LE PONTET,CLINIQUE DU SOUFFLE LE PONTET,None,None,95,01,A,76,15236028.000000,02,6,None,311,R,DE LA CHAPELLE,None,None,01110,PLATEAU D HAUTEVILLE,01185
88889,9,010000255,779309194,ASS.ACCUEIL FORMA INSERT PERSON SOURD,ASSOCIATION POUR LA FORMATION ET L'INSERTION DES PERSONNES,ASS AFIS,None,60,01,A,118,16329016.000000,02,9,None,5,R,DU LYCEE,None,None,01000,BOURG EN BRESSE,01053
7779,17,010000339,260100011,RESIDENCE FONTELUNE,RESIDENCE FONTELUNE,None,None,21,01,A,230,762777.000000,02,17,None,10,R,DE LA COMMUNE 1871,None,None,01500,AMBERIEU EN BUGEY,01004
8890,18,010000347,260100029,LA RESIDENCE D'URFE,LA RESIDENCE D'URFE,None,None,21,01,A,244,762778.000000,02,18,None,305,R,DE L'ECOLE PRIVEE,None,None,01570,FEILLENS,01159


## 2. Etablissements

In [5]:
df_ege = charger_ege(conn, TABLES, actifs_seulement=ACTIFS_SEULEMENT)
print(f"EGE charges : {len(df_ege):,}")

df_ege = dedupliquer(df_ege, "NB_EgeId", usage_prioritaire=None)

print(f"   avec SIRET        : {df_ege['TX_Siret'].notna().sum():,}")
print(f"   avec PM rattachee : {df_ege['NB_PmSmsseId'].notna().sum():,}")
print(f"   avec adresse      : {df_ege['TX_CogCommune'].notna().sum():,}")

df_ege.to_parquet(FINESS_EGE_RAW, index=False)
print(f"Sauvegarde : {FINESS_EGE_RAW}")
afficher_tableau(df_ege, "Apercu EGE FINESS+")

EGE charges : 109,316
   deduplication sur NB_EgeId : 109,316 -> 104,649 lignes
   avec SIRET        : 91,545
   avec PM rattachee : 104,649
   avec adresse      : 104,648
Sauvegarde : /home/jovyan/work/finess+_refonte/projet_finess_sirene_finess+/data/interim/finess_ege.parquet


ID_Ege,NB_EgeId,NB_PmSmsseId,TX_NumFinessEge,TX_Siret,TX_NomEgeLong,TX_NomEgeCourt,TX_ComplementDenominationEg,TX_CategorieEntiteGeographiqueExercice,TX_EtatActif,FK_MetadonneeId,NB_EtablissementInseeId,TX_UsageAdresse,NB_AdresseId,TX_TypeAdresse,TX_NumeroVoie,TX_TypeVoie,TX_LibelleVoie,TX_ComplementPointGeographique,TX_LieuDit,TX_CodePostal,TX_Localite,TX_CogCommune
1,1,580,010000024,26010004500012,CENTRE HOSPITALIER DE BOURG-EN-BRESSE FLEYRIAT,CH DE FLEYRIAT,None,355,A,1374217,1097751.000000,03,98159.000000,None,900,RTE,DE PARIS,None,None,01440,VIRIAT,01451
85688,2,581,010000032,26010003700068,CENTRE HOSPITALIER BUGEY SUD,CH BUGEY SUD,None,355,A,1374238,1097748.000000,03,98161.000000,None,700,AV,DE NARVIK,None,BP 139,01300,BELLEY,01034
119021,5,584,010000065,26010028400017,CENTRE HOSPITALIER DE TREVOUX - MONTPENSIER,CH DE TREVOUX - MONTPENSIER,None,355,A,1374301,1097841.000000,03,98167.000000,None,14,R,DE L'HOPITAL,None,BP 615,01606,TREVOUX CEDEX,01427
141243,7,586,010000081,26010010200011,CENTRE HOSPITALIER DU PAYS DE GEX,CH DU PAYS DE GEX,None,106,A,1374343,1097780.000000,03,98171.000000,None,160,R,MARC PANISSOD,None,BP 437,01174,GEX CEDEX,01173
152354,8,587,010000099,26010013600019,CENTRE HOSPITALIER DE MEXIMIEUX,CH DE MEXIMIEUX,None,106,A,1374364,1097790.000000,03,98173.000000,None,13,AV,DU DOCTEUR BOYER,None,None,01800,MEXIMIEUX,01244


## 3. Dates d'ouverture des personnes morales

FINESS+ ne porte pas de date d'ouverture exploitable : `DT_DateDebutValidite`
est la date de mise a jour de la version. Elle est reconstituee depuis la table
Evenement, en retenant les evenements de code 001 portant sur une PM.

Une structure pouvant en compter plusieurs, on garde la plus ancienne.


In [6]:
df_evt = charger_dates_ouverture_pm(conn, TABLES)
conn.close()

df_evt.to_parquet(EVENEMENTS_PM, index=False)
print(f"Sauvegarde : {EVENEMENTS_PM}")

annees = pd.to_datetime(df_evt["date_ouverture"], errors="coerce").dt.year
print()
print(annees.describe().to_string())
afficher_tableau(df_evt.head(10), "Apercu des dates d'ouverture")


   98,194 PM avec une date d'ouverture, dont 1 avec plusieurs evenements 001
Sauvegarde : /home/jovyan/work/finess+_refonte/projet_finess_sirene_finess+/data/interim/evenements_ouverture_pm.parquet

count    98194.000000
mean      1998.335947
std         20.271739
min       1900.000000
25%       2001.000000
50%       2001.000000
75%       2009.000000
max       2026.000000


finess,date_ouverture,nb_evenements
010000016,1994-01-01 00:00:00,1
010000248,2001-01-01 00:00:00,1
010000412,2001-01-01 00:00:00,1
010000511,2001-01-01 00:00:00,1
010000529,2001-01-01 00:00:00,1
